# 01. 표의 순열 불변성과 In-context 예측

## 목표
표에서 행·열 순서가 의미를 바꾸지 않아야 하는 이유를 확인하고, 레이블이 있는 문맥 행으로 새 행을 예측하는 간단한 모델을 구현합니다. 실제 TabFM이 아니라 핵심 인터페이스를 학습하는 표준 라이브러리 예제입니다.

In [ ]:
from math import sqrt

columns = ["age", "income"]
X_context = [
    {"age": 25.0, "income": 40.0},
    {"age": 45.0, "income": 90.0},
    {"age": 30.0, "income": 50.0},
    {"age": 50.0, "income": 100.0},
]
y_context = [0, 1, 0, 1]
X_test = {"age": 47.0, "income": 92.0}

In [ ]:
def distance(left, right, feature_order):
    # 열 이름으로 값을 찾으므로 feature_order가 바뀌어도 같은 거리가 나옵니다.
    return sqrt(sum((left[name] - right[name]) ** 2 for name in feature_order))

def predict_one(context_rows, context_labels, test_row, feature_order):
    nearest = min(
        zip(context_rows, context_labels),
        key=lambda pair: distance(pair[0], test_row, feature_order),
    )
    return nearest[1]

original = predict_one(X_context, y_context, X_test, columns)
row_permuted = predict_one(list(reversed(X_context)), list(reversed(y_context)), X_test, columns)
column_permuted = predict_one(X_context, y_context, X_test, list(reversed(columns)))
print(original, row_permuted, column_permuted)
assert original == row_permuted == column_permuted

## 해석과 연습

이 예제는 거리가 열 순서에 무관하고, 사례와 label을 함께 재배치하면 행 순서에도 무관합니다. 실제 TabFM은 교대 attention과 행 압축으로 훨씬 복잡한 관계를 표현합니다.

1. `debt` 열을 추가하고 예측이 어떻게 변하는지 확인하세요.
2. 숫자 scale 차이가 거리를 지배하지 않도록 표준화를 추가하세요.
3. 범주형 열을 안전하게 비교하는 방법을 설계하세요.